# Project 4: CDC Pipeline (PostgreSQL Source → Redpanda → Spark → Warehouse)
This notebook demonstrates simulated Change Data Capture (CDC). 
1. We write random updates, inserts, and deletes into a **Source PostgreSQL database**.
2. These actions represent transactional updates. We serialize these mutations as CDC payload events and stream them to **Redpanda** (Azure Event Hub equivalent).
3. We use **Spark Structured Streaming** to consume the CDC topic and dynamically update the **Target PostgreSQL Warehouse** table.

## 1. Establish Source PostgreSQL Database
Let's write a python snippet to establish a source table `source_inventory` in our Postgres database.

In [ ]:
import psycopg2
import os

db_host = os.environ.get('POSTGRES_HOST', 'postgres-dw')
conn = psycopg2.connect(
    host=db_host,
    database="synapse_dw",
    user="postgres",
    password="password",
    port=5432
)
cursor = conn.cursor()

# Create source inventory table
cursor.execute("""
CREATE SCHEMA IF NOT EXISTS source_sys;
CREATE TABLE IF NOT EXISTS source_sys.inventory (
    item_id INT PRIMARY KEY,
    item_name VARCHAR(100),
    quantity INT,
    price DECIMAL(10, 2),
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
conn.commit()

# Insert base inventory rows
cursor.execute("""
INSERT INTO source_sys.inventory (item_id, item_name, quantity, price)
VALUES 
    (1, 'Laptop M4', 50, 1299.99),
    (2, 'iPhone 15', 120, 999.00),
    (3, 'iPad Pro', 40, 799.00)
ON CONFLICT DO NOTHING;
""")
conn.commit()

print("✅ Source inventory database tables initialized.")
cursor.close()
conn.close()

## 2. Simulate Transactional CRUD Updates (CDC Event Producer)
Here we define a python block that modifies values in `source_sys.inventory` and pushes those transaction events (insert, update) to Redpanda. This represents the CDC agent (like Debezium or Synapse Link).

In [ ]:
import json
import random
from datetime import datetime

def generate_cdc_payload(op_type, item_id, item_name, quantity, price):
    return {
        "op": op_type, # 'I' (Insert), 'U' (Update), 'D' (Delete)
        "ts": datetime.now().isoformat(),
        "before": {"item_id": item_id} if op_type == 'U' else None,
        "after": {
            "item_id": item_id,
            "item_name": item_name,
            "quantity": quantity,
            "price": float(price)
        }
    }

print("CDC Generator utility ready.")

## 3. Spark CDC Stream Ingestion & target warehouse synchronization
We use PySpark Structured Streaming to read the CDC events from Redpanda, extract the operations ('op' type), and write them using `foreachBatch` to perform UPSERTS into the target warehouse table `synapse_dw.dim_inventory`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder.appName("CDCPipeline").getOrCreate()

# Read from Redpanda broker
try:
    cdc_stream = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "redpanda:9092") \
        .option("subscribe", "cdc-inventory-events") \
        .option("startingOffsets", "earliest") \
        .load()
    print("✅ CDC Stream connected successfully.")
except Exception as e:
    print(f"Stream connection error: {e}. Check Docker services.")